ITT:  
1. [Connecting to LLM](#connecting-llm)  
2. [Running Sandboxed Environment](#running-sandboxed-environment)  
3. [Testing on MBPP](#testing-on-mbpp)  
    


# Connecting LLM

> [!NOTE]  
> Please don't abuse these services, else we might lose them.

Go to [link](https://huggingface.co/settings/tokens) to regenerate HF_TOKEN.

In [ ]:
# %env HF_TOKEN=asd

In [ ]:
# Set HF_TOKEN for this notebook session (use your token from https://huggingface.co/settings/tokens)
# Option 1: Jupyter magic (persists to all cells)
# %env HF_TOKEN=your_token_here

# Option 2: Python (persists to all cells)
import os
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "your_token_here")
# print(os.environ["HF_TOKEN"])

In [ ]:
# %uv pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

def get_completion(prompt):
    completion = client.chat.completions.create(
        model="moonshotai/Kimi-K2-Instruct-0905",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
)
    return completion.choices[0].message.content



In [ ]:
print(get_completion("Tell me a joke"))

---- Test done ----

# Running Sandboxed Environment

> We decided to explore Firecracker, docker and microVMs and decided to use Docker for now as Firecracker is overkill

Build the image (from `agent-smith/`):

```bash
docker build -f DOCKER/Dockerfile -t my-agent-image .
```

Use:
```bash
uv run --active sandbox
```

In [ ]:
!docker run --rm \
  --memory=512m \
  --cpus=1 \
  --pids-limit=128 \
  --read-only \
  --cap-drop=ALL \
  --security-opt=no-new-privileges \
  --network=none \
  my-agent-image


Use:
```bash
uv run --active sandbox sandbox_template.json
```

```json
// sandbox_template.json

```

Use:  
```bash
uv run sandbox --mcp-server <URL>
```


# Testing on MBPP

> Assignment wants to have tests. 

```bash
# 1. Dump a task
cd moulinette_master
uv run moulinette_eval dump mbpp --output ../cache/mbpp_task.json
# 2. Run your agent
cd ../student
uv run python -m agent_mbpp --task-file ../cache/mbpp_task.json \
--output ../cache/mbpp_solution.json
# 3. Validate solution
cd ../moulinette_master
uv run moulinette_eval validate mbpp ../cache/mbpp_task.json \
../cache/mbpp_solution.json
```

Testing on SWE

```bash
# 1. Dump a task
cd moulinette_master
uv run moulinette_eval dump swebench --output ../cache/swebench_task.json
# 2. Run your agent
cd ../student
uv run python -m agent_swebench --task-file ../cache/swebench_task.json --
output ../cache/swebench_solution.json
# 3. Validate solution
cd ../moulinette_master
uv run moulinette_eval validate swebench ../cache/swebench_task.json ../cache/
swebench_solution.json
```

# Building Agents
#### BUILDING Agent_mbpp

Agents Parts:  
1. [mbpp agent](#building-agent_mbpp)
1. [swe agent](#building-agent_swe)

autonomous agent dedicated to solving Mostly Basic Python Problems

### 1. Dump a random task

In [ ]:
!cd moulinette_master && uv run moulinette_eval dump mbpp --output ../cache/mbpp_task.json

### 2. Run the agent on task

In [ ]:
!cd student && uv run python -m agent_mbpp --task-file ../cache/mbpp_task.json --output ../cache/mbpp_solution.json

### 3. Validate the solution

In [ ]:
!cd moulinette_master && uv run moulinette_eval validate mbpp ../cache/mbpp_task.json ../cache/mbpp_solution.json

```
Task loading  
Agent execution  
```

### BUILDING agent_swe

> This part focuses on implementing an autonomous agent capable of solving SWE-bench tasks inside Dockerized environments.  
> Usages:  
>   1. Fix real bugor implement features in real repositories
>   2. Explore codebases inside Docker containers, you are responsible to clean it after your program execution
>   3. Generate and submit valid patches using ’git -c core.fileMode=false diff’

```python
#task input
class SWEBenchTaskInput(BaseModel):
    """Input for SWE-bench task evaluation.
    You are responsible for pulling and managing the Docker container.
    The docker_image field contains the full image name to pull.
    The eval_script is used to run tests inside the container.
    """
    instance_id: str
    repo: str = ""
    docker_image: str # Full image name, e.g., "swebench/sweb.eval.x86_64.
    sympy_1776_sympy-23534:latest"
    problem_statement: str
    hints_text: str = ""
    eval_script: str # Bash script to run tests inside the container
```

The bellow part contains a "patch"...

##### EXAMPLE:

SWE-bench is a benchmark for evaluating autonomous coding agents. It tests whether an agent can fix real GitHub issues from real repositories.

- It must be a valid code change  
- Written as a unified diff (`diff` / git diff format)  
- Modifies the repository files  
- No extra commentary — just the patch  

Example structure:
```diff
diff --git a/file.py b/file.py
index 123..456 100644
--- a/file.py
+++ b/file.py
@@ -10,7 +10,7 @@
-    return x + 1
+    return x + 2
```

#### class templates:

```python
#agent output
class StepMetrics(BaseModel):
    """Metrics for a single agent step."""
    step: int
    input_tokens: int
    output_tokens: int
    request_time_ms: float
    timestamp: str = Field(default_factory=lambda: datetime.now().
    isoformat())

class SolutionOutput(BaseModel):
    """Result of your solution: this is what you need to produce."""
    task_id: str
    benchmark: str # "mbpp" or "swebench"
    success: bool
    solution: str # Code for MBPP, patch for SWE-bench
    iterations: int
    total_requests: int
    total_input_tokens: int
    total_output_tokens: int
    total_time_seconds: float
    steps: List["StepMetrics"] = Field(default_factory=list)
    error: Optional[str] = None
    timestamp: str = Field(default_factory=lambda: datetime.now().
    isoformat())
```

### 1. Dump a random task

In [63]:
!cd moulinette_master && uv run moulinette_eval dump swebench --output ../cache/swebench_task.json

2026-02-15 16:35:52,540 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-02-15 16:35:52,670 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Verified/99450355ca8c611021187a57ffac304b66666738/README.md "HTTP/1.1 200 OK"
2026-02-15 16:35:52,912 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/SWE-bench/SWE-bench_Verified/99450355ca8c611021187a57ffac304b66666738/README.md "HTTP/1.1 200 OK"
README.md: 3.34kB [00:00, 2.78MB/s]
2026-02-15 16:35:53,085 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/SWE-bench/SWE-bench_Verified/resolve/99450355ca8c611021187a57ffac304b66666738/SWE-bench_Verified.py "HTTP/1.1 404 Not Found"
2026-02-15 16:35:53,495 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SWE-bench/SWE-bench_Verifie

### 2. Run the agent on task

In [65]:
!cd ./student && uv run python -m agent_swebench --task-file ../cache/swebench_task.json --output ../cache/swebench_solution.json

/Users/sharkattack/Documents/GitHub/totally-reasonable-42-ai/agent-smith/.venv/bin/python3: No module named agent_swebench


### 3. Validate the task completion

In [67]:
!cd ./moulinette_master && uv run moulinette_eval validate swebench ../cache/swebench_task.json ../cache/swebench_solution.json

Traceback (most recent call last):
  File "/Users/sharkattack/Documents/GitHub/totally-reasonable-42-ai/agent-smith/moulinette_master/.venv/bin/moulinette_eval", line 10, in <module>
    sys.exit(main())
  File "/Users/sharkattack/Documents/GitHub/totally-reasonable-42-ai/agent-smith/moulinette_master/moulinette_eval/__main__.py", line 244, in main
    passed = cmd_validate(args)
  File "/Users/sharkattack/Documents/GitHub/totally-reasonable-42-ai/agent-smith/moulinette_master/moulinette_eval/__main__.py", line 75, in cmd_validate
    with open(solution_path) as f:
FileNotFoundError: [Errno 2] No such file or directory: '../cache/swebench_solution.json'


--------------------- footer ---------------------

Leveraging:
1. uv for package managing
2. hugging face for api
3. openai module